# Introduction to ecological data analysis
This notebook uses the completely synthetic potato-soil dataset. It shows how a community table and metadata work together.

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import spearmanr

data = Path('../Data/Synthetic_Potato_Example')
counts = pd.read_csv(data / 'ITS_OTU_table_final.tsv', sep='\t', index_col=0)
metadata = pd.read_csv(data / 'Final_data_with_diversity_prefixed.csv').set_index('sample_id')
counts.head()

## Reads and richness
Each row is a soil sample and each column is a fungal species hypothesis. Sequencing depth is the row sum. Richness is the number of taxa with at least one read.

In [ ]:
total_reads = counts.sum(axis=1)
richness = (counts > 0).sum(axis=1)
summary = metadata.join(pd.DataFrame({'total_reads': total_reads, 'observed_richness': richness}))
summary[['total_reads', 'observed_richness', 'pH_KCl']].head()

## Why depth matters
Low-depth samples often miss rare taxa, so compare depth and observed richness before interpreting richness differences.

In [ ]:
plt.scatter(summary.total_reads, summary.observed_richness, alpha=.7)
plt.xlabel('ITS sequencing reads'); plt.ylabel('Observed richness'); plt.show()
rho, p = spearmanr(summary.total_reads, summary.observed_richness)
print(f'Spearman rho = {rho:.2f}, p = {p:.3g}')

## Shannon diversity
Shannon diversity combines richness and evenness. Empty libraries are excluded because proportions are undefined.

In [ ]:
nonempty = counts.loc[total_reads > 0]
proportions = nonempty.div(nonempty.sum(axis=1), axis=0)
shannon = -(proportions.where(proportions > 0, 1) * proportions.where(proportions > 0, 1).apply(lambda x: x.map(__import__('math').log))).sum(axis=1)
summary.loc[shannon.index, 'shannon'] = shannon
summary[['pH_KCl', 'shannon']].dropna().plot.scatter(x='pH_KCl', y='shannon')
plt.show()

Interpret associations as teaching examples only. The synthetic README describes the deliberately simulated relationships.